# Adversarial TTS - Class-Based Architecture (Google Colab)

This notebook uses the refactored class-based architecture:
- **EnvironmentLoader**: Handles model loading and environment setup
- **AdversarialTrainer**: Runs the optimization loop (returns results)
- **RunLogger**: Handles all output and logging (called separately)

## Setup
Run the cell below to clone the repository and install dependencies.

In [ ]:
%%bash
git clone https://github.com/Vorgesetzter/StyleTTS2
cd StyleTTS2
pip install -r requirements.txt
sudo apt-get install espeak-ng
git-lfs clone https://huggingface.co/yl4579/StyleTTS2-LJSpeech
mkdir -p Audio
mv StyleTTS2-LJSpeech/Models/* Audio/
rm -rf StyleTTS2-LJSpeech

## 1. Imports

In [ ]:
import torch
import os
import argparse
import matplotlib.pyplot as plt

os.chdir("/content/StyleTTS2")

from Datastructures.enum import AttackMode
from Datastructures.dataclass import ModelData
from Objectives.FitnessObjective import FitnessObjective

# Import class-based modules
from Trainer.EnvironmentLoader import EnvironmentLoader
from Trainer.AdversarialTrainer import AdversarialTrainer
from Trainer.RunLogger import RunLogger
from Trainer.GraphPlotter import GraphPlotter
from Trainer.VectorManipulator import VectorManipulator

# Import Pymoo components
from Optimizer.pymoo_optimizer import PymooOptimizer
from pymoo.algorithms.moo.nsga2 import NSGA2

from helper import write_run_summary

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Available GPUs: {torch.cuda.device_count()}")

## 2. Configuration

Edit the values below to configure the optimization run.

In [ ]:
# =============================================================================
# Configuration - Edit these values directly
# =============================================================================
args = argparse.Namespace(
    # Text parameters
    ground_truth_text="I think the NFL is lame and boring",
    target_text="The Seattle Seahawks are the best Team in the world",

    # Optimization parameters
    loop_count=1,
    num_generations=100,
    pop_size=200,
    iv_scalar=0.5,
    size_per_phoneme=1,
    batch_size=100,  # -1 for full batch

    # Flags
    notify=False,
    subspace_optimization=False,

    # Mode and objectives
    mode="TARGETED",  # TARGETED, UNTARGETED, NOISE_UNTARGETED
    ACTIVE_OBJECTIVES=["PESQ", "WER_GT"],
    thresholds=["PESQ=0.3", "WER_GT=0.5"],
)

## 3. Initialize Environment

Load models and prepare audio data.

In [ ]:
# Initialize environment using EnvironmentLoader
loader = EnvironmentLoader(device)
config_data = loader.load_configuration(args)
config_data.print_summary()

tts_model, asr_model = loader.load_required_models()

audio_gt, audio_target, audio_embedding_gt, audio_embedding_target = loader.generate_audio_data(
    config_data.mode, config_data.text_gt, config_data.text_target, tts_model
)

print("\nEnvironment initialized successfully!")

## 4. Setup Trainer Components

Initialize ObjectiveManager, VectorManipulator, Trainer, and Logger.

In [ ]:
# Initialize Objectives
objectives = loader.initialize_objectives(
    active_objectives=config_data.active_objectives,
    model_data=ModelData(tts_model=tts_model, asr_model=asr_model),
    text_gt=config_data.text_gt,
    text_target=config_data.text_target,
    mode=config_data.mode,
    audio_gt=audio_gt,
)

vector_manipulator = VectorManipulator(audio_embedding_gt, audio_embedding_target.h_text, config_data)

# Create Trainer and Logger
trainer = AdversarialTrainer(
    tts_model, asr_model, config_data.active_objectives,
    config_data.thresholds, objectives, vector_manipulator, device
)
logger = RunLogger(
    config_data.active_objectives, tts_model, asr_model, vector_manipulator, device
)

print("Trainer components initialized!")

## 5. Run Optimization

Run the optimization loop and log results.

In [ ]:
# Run optimization loops
for loop_iteration in range(config_data.loop_count):
    print(f"\n[Loop {loop_iteration + 1}/{config_data.loop_count}] Starting optimization loop...")

    # Initialize fresh optimizer for this cycle
    optimizer = PymooOptimizer(
        bounds=(0, 1),
        algorithm=NSGA2,
        algo_params={"pop_size": config_data.pop_size},
        num_objectives=len(config_data.active_objectives),
        solution_shape=(audio_embedding_gt.input_length.detach().cpu().item(), config_data.size_per_phoneme),
    )

    fitness_data, generation_count, elapsed_time_total = trainer.run_full_iteration(
        optimizer, config_data.num_generations, config_data.pop_size, config_data.batch_size
    )

    # Log Results
    folder_path = logger.setup_output_directory()
    logger.save_fitness_history(fitness_data)

    best_candidate = logger.select_best_candidate(optimizer.best_candidates, config_data.thresholds)
    audio_best, text_best, audio_embedding_best = logger.run_final_inference(best_candidate)

    logger.save_audios(audio_gt, audio_target, audio_best)
    logger.save_torch_state(text_best, audio_embedding_best, best_candidate, config_data)

    # Generate Graphs
    graph_plotter = GraphPlotter(config_data.active_objectives, generation_count, folder_path, fitness_data)
    graph_plotter.generate_hypervolume_graph()
    graph_plotter.generate_pareto_population_graph()
    graph_plotter.generate_mean_population_graph()
    graph_plotter.generate_minimal_population_graph()
    plt.close('all')

    # Write Summary
    write_run_summary(folder_path, text_best, best_candidate, generation_count, elapsed_time_total, config_data)

    print("[Log] Finished saving all results")

## 6. Download Results (Optional)

In [ ]:
extract = False  # Set to True to download results

if extract:
    !(cd /content/StyleTTS2 && zip -r /content/colab_results.zip outputs/)

    from google.colab import files
    files.download('/content/colab_results.zip')